# 모델 검증 — 시간 기반 train/test 분리

**모델 C/A의 성능이 '미래 신작'에도 유지되는가**를 확인하는 노트북입니다.

- 분할: 작품별 첫 Top 10 진입일 기준 (2021-11-01 컷) — 무작위 분할과 달리 시간의 흐름을 반영
- 산출물: `data/interim/title_time_split.csv`(분할 구성표), `data/processed/model_validation_time_split.csv`(검증 결과)

In [1]:
# 모델 검증: 시간 기반 train/test 분리
# 설계: 작품별 "첫 Top 10 진입일" 기준. 2021-11-01 이전 진입(과거) = 학습, 이후 진입(신작) = 테스트
# 실무 시나리오 "미래 신작 예측"에 가장 가까운 평가 방식

from pathlib import Path
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import roc_auc_score, mean_absolute_error, r2_score

ROOT = Path(r"C:\Users\USER\Desktop\엘리스 개인 프로젝트 자료\[엔터] Netflix Top 10 Weekly Dataset")
ORDER = ["유럽", "아시아", "북중미", "남미", "아프리카", "오세아니아"]

cm = pd.read_csv(ROOT / "references/country_continent.csv")
CONT = dict(zip(cm.country_name, cm.대륙))
c = pd.read_csv(ROOT / "data/raw/all-weeks-countries.csv", encoding="cp1252", parse_dates=["week"])
w = pd.read_csv(ROOT / "data/raw/all-weeks-global.csv", parse_dates=["week"])
pres = pd.read_csv(ROOT / "data/processed/title_continent_presence.csv")
c["대륙"] = c.country_name.map(CONT)

first = c.groupby("show_title").week.min().rename("first_week")
c = c.merge(first, on="show_title")
c["offset_wk"] = ((c.week - c.first_week).dt.days // 7).astype(int)
cat = c.groupby("show_title").category.agg(lambda s: s.mode()[0]).rename("category")
first_cont = c.sort_values(["show_title", "week", "country_name"]).groupby("show_title").대륙.first()
titles = c.show_title.unique()

In [2]:
# 발자국 스냅샷 생성 (작품별 첫 진입 후 경과 t주까지의 누적 정보)
def snapshot(t):
    sub = c[c.offset_wk < t]
    g = sub.groupby("show_title")
    f = pd.DataFrame(index=pd.Index(titles, name="show_title"))
    f["진입국가수"] = g.country_name.nunique().reindex(f.index).fillna(0).astype(int)
    f["진입대륙수"] = g.대륙.nunique().reindex(f.index).fillna(0).astype(int)
    pv = (sub.groupby(["show_title", "대륙"]).country_name.nunique().unstack("대륙")
          .reindex(columns=ORDER).fillna(0).astype(int).reindex(f.index).fillna(0).astype(int))
    pv.columns = [f"{k}국가수" for k in ORDER]
    f = f.join(pv)
    for k in ORDER:
        f[f"첫진입_{k}"] = (first_cont.reindex(f.index) == k).astype(int)
    f["TV여부"] = (cat.reindex(f.index) == "TV").astype(int)
    return f

## 1. 시간 분할 구성표

In [3]:
# 시간 분할 + 분할 구성표 저장
CUT = pd.Timestamp("2021-11-01")
first_map = first.reindex(titles)
is_late = (first_map >= CUT).to_numpy()

y_global = np.array([t in set(w.show_title.unique()) for t in titles], dtype=int)
y_reach = pres.groupby("show_title").등장국가수.max().reindex(titles).to_numpy().astype(float)

split = pd.DataFrame({"show_title": titles, "first_week": first_map.to_numpy(),
                      "구간": np.where(is_late, "테스트(11월 이후 신작)", "학습(10월 이전)"),
                      "글로벌히트여부": y_global, "최종진입국가수": y_reach})
split.to_csv(ROOT / "data/interim/title_time_split.csv", index=False, encoding="utf-8-sig")
print(f"학습 {(~is_late).sum()}편 / 테스트 {is_late.sum()}편 -> data/interim/title_time_split.csv 저장")

학습 1401편 / 테스트 605편 -> data/interim/title_time_split.csv 저장


## 2. 검증 실행 (분류 AUC / 회귀 MAE·R²)

In [4]:
# 시간 기반 검증 실행 (스냅샷 1주차/4주차)
rows = []
for t in [1, 4]:
    X = snapshot(t).to_numpy().astype(float)
    clf = LogisticRegression(max_iter=3000, C=1.0, class_weight="balanced").fit(X[~is_late], y_global[~is_late])
    p = clf.predict_proba(X[is_late])[:, 1]
    rg = Ridge(alpha=1.0).fit(X[~is_late], y_reach[~is_late])
    pr = np.clip(rg.predict(X[is_late]), 0, 94)
    rows.append({"스냅샷": f"{t}주차",
                 "분류_AUC": round(roc_auc_score(y_global[is_late], p), 3),
                 "회귀_MAE_개국": round(mean_absolute_error(y_reach[is_late], pr), 2),
                 "회귀_R2": round(r2_score(y_reach[is_late], pr), 3),
                 "테스트편수": int(is_late.sum())})
res = pd.DataFrame(rows)
res.to_csv(ROOT / "data/processed/model_validation_time_split.csv", index=False, encoding="utf-8-sig")
print("=== 시간 기반 검증 결과 (과거 학습 -> 신작 테스트) ===")
print(res.to_string(index=False))
print("-> data/processed/model_validation_time_split.csv 저장")
print()
print("(비교) 무작위 5-fold CV: 4주차 분류 AUC 0.905, 회귀 R2 0.898")
print("해석: 시간 분할에서도 성능이 유지되어 미래 신작에 대한 일반화 가능")

=== 시간 기반 검증 결과 (과거 학습 -> 신작 테스트) ===
스냅샷  분류_AUC  회귀_MAE_개국  회귀_R2  테스트편수
1주차   0.896       4.36  0.864    605
4주차   0.932       2.25  0.978    605
-> data/processed/model_validation_time_split.csv 저장

(비교) 무작위 5-fold CV: 4주차 분류 AUC 0.905, 회귀 R2 0.898
해석: 시간 분할에서도 성능이 유지되어 미래 신작에 대한 일반화 가능
